# 05 — Classify the OOM Without Summoning One

**Target:** Apache Spark 4.2.0; Python 3.10+.

Run cells in order. These labs use small local workloads and avoid deliberately exhausting cluster resources. Read each *Why* before changing the experiment.

## Question
Can we collect evidence to distinguish JVM heap pressure, native thread limits, Python worker memory and a kernel kill without causing an OOM?

**Why this matters:** the label “OOM” hides failures with different owners and remedies.

In [ ]:
from pyspark.sql import SparkSession
import os, pathlib, subprocess, sys
spark=SparkSession.builder.master('local[2]').appName('thread-lab-05').getOrCreate()
print('Spark',spark.version,'Python',sys.version.split()[0],'driver PID',os.getpid())
for key in ['spark.executor.memory','spark.executor.memoryOverhead','spark.executor.pyspark.memory','spark.python.worker.memory']:
    print(key, spark.sparkContext.getConf().get(key,'<unset>'))
print('Interpretation: <unset> means this SparkConf has no explicit value. It is not the effective default; defaults and deploy-time values depend on Spark version and cluster manager.')


In [ ]:
# Resolve this notebook Python process's cgroup v2 path via /proc/self/cgroup + mountinfo.
from pathlib import Path, PurePosixPath
import os
proc_cgroup=Path('/proc/self/cgroup')
mountinfo=Path('/proc/self/mountinfo')
if proc_cgroup.exists() and mountinfo.exists():
    membership=None
    for line in proc_cgroup.read_text().splitlines():
        hierarchy, controllers, path = line.split(':',2)
        if hierarchy == '0' and controllers == '': membership=PurePosixPath(path); break
    target=None
    def unescape_mount(value):
        return value.replace('\\040',' ').replace('\\011','\t').replace('\\134','\\')
    if membership is not None:
        for line in mountinfo.read_text().splitlines():
            before, after=line.split(' - ',1)
            post=after.split()
            if post[0] != 'cgroup2': continue
            fields=before.split(); mount_root=PurePosixPath(unescape_mount(fields[3])); mount_point=Path(unescape_mount(fields[4]))
            try: relative=membership.relative_to(mount_root)
            except ValueError: continue
            target=mount_point.joinpath(*relative.parts); break
    print('Notebook process cgroup:',membership)
    print('Resolved cgroup v2 directory:',target)
    if target and target.exists():
        for name in ['memory.current','memory.max','memory.events','pids.current','pids.max','pids.events']:
            path=target/name
            print(f'{name}:',path.read_text().strip() if path.exists() else '<not exposed>')
    else:
        print('No matching cgroup v2 mount found; cgroup metrics unavailable.')
    print('Scope: this is the notebook Python kernel cgroup. The local Spark JVM usually shares it, but verify process membership; inspect each remote executor container separately.')
else:
    print('Linux /proc cgroup v2 metadata unavailable on this operating system.')

## Evidence matrix
- `Java heap space`: inspect GC logs and heap dump/dominators.
- `SparkOutOfMemoryError`: capture the Spark condition/error class (for example, `UNABLE_TO_ACQUIRE_MEMORY`, `AGGREGATE_OUT_OF_MEMORY`, `POINTER_ARRAY_OUT_OF_MEMORY`, `SPILL_OUT_OF_MEMORY`) and task attempt; inspect execution-memory acquisition and spill behavior. This is a Spark task-level memory-manager failure, not proof that the JVM heap or whole container is exhausted. See [Spark 4.2 error conditions](https://spark.apache.org/docs/4.2.0/sql-error-conditions.html) and [the SparkOutOfMemoryError class](https://github.com/apache/spark/blob/v4.2.0/core/src/main/java/org/apache/spark/memory/SparkOutOfMemoryError.java).
- `unable to create native thread`: inspect JVM thread dump, OS/cgroup PID limits and native resources.
- `OOMKilled`: correlate the process container's cgroup `memory.events`, orchestrator events and process/container memory.
- Python worker exit: correlate worker logs/process RSS and executor container budget.
- Direct buffer failure: inspect direct-buffer use and exact JVM message.

**Expected:** configuration and cgroup values describe the notebook Python kernel's environment; they do not prove a cause or automatically represent a remote Spark executor.

**Why:** NMT must be enabled before the JVM starts and does not count separate Python processes. `memoryOverhead` is not a heap partition; PySpark memory limiting is platform dependent. Avoid `-Xmx × thread count` style arithmetic as a measure of resident RAM.

In [ ]:
spark.stop()